In [1]:
import os
import sys
import random
from astropy.io import fits
import numpy as np
import pickle
import matplotlib.pyplot as plt

In [ ]:
def expand_points(wavelength, flux, target_count):
    wl = list(wavelength)
    fl = list(flux)
    diffs = [abs(fl[i+1] - fl[i]) for i in range(len(fl)-1)]
    sorted_indices = sorted(range(len(diffs)), key=lambda i: diffs[i], reverse=True)
    
    while len(wl) < target_count:
        for idx in sorted_indices:
            if len(wl) >= target_count:
                break
            new_wl = (wl[idx] + wl[idx+1]) / 2
            new_fl = (fl[idx] + fl[idx+1]) / 2
            wl.insert(idx+1, new_wl)
            fl.insert(idx+1, new_fl)
    
    return np.array(wl), np.array(fl)

# Cargar la lista estática de archivos FITS
files_list_file = 'extra/files_list.pkl'
with open(files_list_file, 'rb') as f:
    files = pickle.load(f)

# Ruta a la carpeta con los archivos FITS a procesar
folder_path = r'spectrums'
batch_size = 10000  # Número de archivos a procesar por lote
index_file = "extra/batch_index_i.txt"

# Verificar que el archivo exista y contenga un número entero
if not os.path.exists(index_file):
    print(f"Error: El archivo {index_file} no existe. Deteniendo la ejecución.")
    sys.exit(1)

with open(index_file, "r") as f:
    content = f.read().strip()
    try:
        batch_index = int(content)
    except ValueError:
        print(f"Error: El archivo {index_file} no contiene un número entero válido. Deteniendo la ejecución.")
        sys.exit(1)

# Calcular los índices del lote actual (recorrer 10k archivos)
start_index = (batch_index - 1) * batch_size
# end_index es el mínimo entre el cálculo y la longitud total de la lista
end_index = min(batch_index * batch_size - 1, len(files))

# Finalización
if batch_index > 270:
    print("Hemos procesado los 300,000 del lote.")
    sys.exit(0)

if start_index >= len(files):
    print("Todos los archivos han sido procesados.")
    sys.exit(0)

# Seleccionar el lote actual (serán 10,000 archivos, o menos si es el último lote)
current_batch = files[start_index:end_index]

spectra_data = {}
i = 0

for filename in current_batch:
    file_path = os.path.join(folder_path, filename)
    try:
        with fits.open(file_path) as hdul:
            # Verificar que el archivo tenga las extensiones esperadas
            if len(hdul) > 2:
                flux_data = hdul[1].data["flux"]         # Datos de flujo
                loglam_data = hdul[1].data["loglam"]        # Datos de log(lambda)
                wavelength_data = 10 ** loglam_data         # Convertir log(lambda) a longitud de onda
                redshift = hdul[2].data["Z"][0]             # Extraer redshift
                
                # Aplicar la interpolación para obtener 5000 puntos
                interp_wavelength, interp_flux = expand_points(wavelength_data, flux_data, target_count=5000)
                
                # Guardar los datos en el diccionario
                spectra_data[filename] = {
                    "wavelength": interp_wavelength,
                    "flux": interp_flux,
                    "redshift": redshift
                }
                
                i += 1
                if i % 1000 == 0:
                    print(f"Procesado {filename} ({i})")
    except Exception as e:
        print(f"Error procesando {filename}: {e}")

print(f"Se procesaron {len(spectra_data)} archivos FITS en el lote {batch_index}.")

# Guardar el diccionario con los datos procesados en un archivo pickle
output_file = f'data/spectra_data_complete{batch_index}.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(spectra_data, f)
print(f"Datos guardados en {output_file}")

# Guardar la lista de archivos procesados en este lote
batch_list_file = f'extra/batch_files_list{batch_index}.pkl'
with open(batch_list_file, 'wb') as f:
    pickle.dump(current_batch, f)
print(f"Lista de archivos del lote {batch_index} guardada en {batch_list_file}")

# Actualizar el archivo batch_index_a.txt para la siguiente iteración
batch_index += 1
with open(index_file, "w") as f:
    f.write(str(batch_index))

print("Proceso completado para este lote. Para el siguiente lote, se procesarán los archivos desde el índice",
      f"{(batch_index - 1) * batch_size} hasta {min(batch_index * batch_size -1, len(files))}.")